<a href="https://colab.research.google.com/github/Susanta2006/Colab/blob/main/Prisoners_Dilemma_Simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
# If you are running this locally, make sure to install gradio: pip install gradio
import gradio as gr
import matplotlib.pyplot as plt

# Step 4: Create Payoff Logic
def get_scores(move1, move2):
    """Calculates the scores based on the Prisoner's Dilemma payoff matrix."""
    if move1 == "Cooperate" and move2 == "Cooperate":
        return 3, 3
    elif move1 == "Betray" and move2 == "Cooperate":
        return 5, 0
    elif move1 == "Cooperate" and move2 == "Betray":
        return 0, 5
    else: # Betray and Betray
        return 1, 1

# Step 5: Create Simple AI Strategies (Including Bonus Strategies)
def ai_move(strategy, opponent_last_move=None):
    """Determines the move based on the selected strategy."""
    if strategy == "Always Cooperate":
        return "Cooperate"
    elif strategy == "Always Betray":
        return "Betray"
    elif strategy == "Mostly Cooperate":
        # 80% chance to cooperate, 20% to betray
        return random.choices(["Cooperate", "Betray"], weights=[80, 20], k=1)[0]
    elif strategy == "Mostly Betray":
        # 20% chance to cooperate, 80% to betray
        return random.choices(["Cooperate", "Betray"], weights=[20, 80], k=1)[0]
    elif strategy == "Tit-for-Tat":
        # Cooperate on the first move, then copy the opponent's last move
        if opponent_last_move is None:
            return "Cooperate"
        return opponent_last_move
    elif strategy == "Human":
        return "Human" # Placeholder, actual move handled in main loop
    else: # Default Random
        return random.choice(["Cooperate", "Betray"])

# Step 6: Create Main Simulation Function
def play_game(player1_strategy, player2_strategy, rounds, p1_human_move="Cooperate", p2_human_move="Cooperate"):
    """Runs the simulation and generates results and charts."""
    score1 = 0
    score2 = 0
    result = ""

    # Data for Score Charts (Bonus Feature 2)
    p1_scores = [0]
    p2_scores = [0]
    p1_moves = {"Cooperate": 0, "Betray": 0}
    p2_moves = {"Cooperate": 0, "Betray": 0}

    # Track last moves for strategies like 'Tit-for-Tat'
    p1_last_move = None
    p2_last_move = None

    for i in range(int(rounds)):
        # Get moves for this round, checking for Human mode (Bonus Feature 1)
        move1 = p1_human_move if player1_strategy == "Human" else ai_move(player1_strategy, p2_last_move)
        move2 = p2_human_move if player2_strategy == "Human" else ai_move(player2_strategy, p1_last_move)

        # Calculate points
        points1, points2 = get_scores(move1, move2)

        # Update scores
        score1 += points1
        score2 += points2

        # Track history for charts
        p1_scores.append(score1)
        p2_scores.append(score2)
        p1_moves[move1] += 1
        p2_moves[move2] += 1

        # Update history for the next round
        p1_last_move = move1
        p2_last_move = move2

        # Format the output log for this round
        result += f"Round {i + 1}\n"
        result += f"Player A ({player1_strategy}): {move1}\n"
        result += f"Player B ({player2_strategy}): {move2}\n"
        result += f"Scores: {points1} - {points2}\n"
        result += "-" * 20 + "\n"

    # Format the final scores
    result += "\nFinal Scores\n"
    result += f"Player A Total: {score1}\n"
    result += f"Player B Total: {score2}\n\n"

    # Bonus Feature #4: Add Winner Display
    if score1 > score2:
        result += "🏆 Winner: Player A Wins! 🏆"
    elif score2 > score1:
        result += "🏆 Winner: Player B Wins! 🏆"
    else:
        result += "🤝 Result: It's a Tie! 🤝"

    # Generate Score Progression Chart
    fig_scores, ax_scores = plt.subplots(figsize=(6, 4))
    ax_scores.plot(range(int(rounds) + 1), p1_scores, label="Player A", marker='o')
    ax_scores.plot(range(int(rounds) + 1), p2_scores, label="Player B", marker='s')
    ax_scores.set_title("Score Progression")
    ax_scores.set_xlabel("Round")
    ax_scores.set_ylabel("Total Score")
    ax_scores.legend()
    ax_scores.grid(True)
    plt.close(fig_scores) # Close to prevent display in standard output

    # Generate Cooperation vs Betrayal Frequency Chart
    fig_freq, ax_freq = plt.subplots(figsize=(6, 4))
    labels = ['P1 Cooperate', 'P1 Betray', 'P2 Cooperate', 'P2 Betray']
    values = [p1_moves["Cooperate"], p1_moves["Betray"], p2_moves["Cooperate"], p2_moves["Betray"]]
    colors = ['#4CAF50', '#F44336', '#81C784', '#E57373']
    ax_freq.bar(labels, values, color=colors)
    ax_freq.set_title("Move Frequency")
    ax_freq.set_ylabel("Count")
    plt.close(fig_freq)

    return result, fig_scores, fig_freq

# Available strategies list
strategies = [
    "Human",
    "Always Cooperate",
    "Always Betray",
    "Random",
    "Tit-for-Tat",
    "Mostly Cooperate",
    "Mostly Betray"
]

# Step 7: Create Gradio Interface (Upgraded to Blocks for extra features)
with gr.Blocks(title="Prisoner's Dilemma Strategy Simulator") as app:
    gr.Markdown("# Prisoner's Dilemma Strategy Simulator")
    gr.Markdown("Compare game theory strategies, play against AI, and visualize score progression!")

    with gr.Row():
        with gr.Column():
            p1_strat = gr.Dropdown(choices=strategies, value="Human", label="Player A Strategy")
            p1_human_move = gr.Radio(choices=["Cooperate", "Betray"], value="Cooperate", label="Player A Manual Move")

        with gr.Column():
            p2_strat = gr.Dropdown(choices=strategies, value="Random", label="Player B Strategy")
            p2_human_move = gr.Radio(choices=["Cooperate", "Betray"], value="Cooperate", label="Player B Manual Move", visible=False)

    rounds_slider = gr.Slider(minimum=1, maximum=50, step=1, value=5, label="Rounds")

    # Dynamic visibility for Human Move options
    def update_human_move_visibility(strat):
        return gr.update(visible=(strat == "Human"))

    p1_strat.change(update_human_move_visibility, inputs=p1_strat, outputs=p1_human_move)
    p2_strat.change(update_human_move_visibility, inputs=p2_strat, outputs=p2_human_move)

    simulate_btn = gr.Button("Run Simulation", variant="primary")

    with gr.Row():
        output_text = gr.Textbox(label="Simulation Results", lines=15)
        with gr.Column():
            plot_scores = gr.Plot(label="Score Progression Chart")
            plot_freq = gr.Plot(label="Cooperation vs Betrayal Frequency")

    simulate_btn.click(
        fn=play_game,
        inputs=[p1_strat, p2_strat, rounds_slider, p1_human_move, p2_human_move],
        outputs=[output_text, plot_scores, plot_freq]
    )

# Step 8: Launch Application
if __name__ == "__main__":
    # If using Google Colab, Gradio will automatically display inline.
    app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://30f9a6011afddd79ab.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
